# 07 - Deep Agent com subagentes: contexto isolado e compartilhado

Um **subagente** é um agente menor que o agente principal chama pela ferramenta `task`. Ele roda com a própria conversa. Quando termina, devolve só a resposta final ao principal.

A pergunta central é: **o que o subagente enxerga?**

- **Contexto isolado**: o subagente recebe apenas o texto da tarefa que o principal escreveu no `task`. Ele não vê o histórico do principal nem as chamadas que o principal fez antes.
- **Contexto compartilhado**: o principal grava informação em um lugar que o subagente também lê, aqui o sistema de arquivos do deep agent. O subagente lê o arquivo e usa o conteúdo, sem depender do texto da tarefa.

Este notebook roda a mesma pergunta nos dois modos:

1. **Isolado**: o principal só delega. O subagente `pesquisador` consulta os dados sozinho.
2. **Compartilhado**: o principal consulta os dados, grava um resumo em `/compartilhado/vendas.md` e delega a redação ao subagente `redator`, que lê o arquivo.

O que comparar: quantas chamadas de ferramenta aparecem no contexto do principal, e de onde vêm os dados usados na resposta.

Tudo fica registrado no MLflow, no experimento `deep-agent-subagentes`.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

O rastreamento usa o MLflow com um banco SQLite, `mlflow.db`, criado na pasta deste notebook.

In [1]:
import ast
import json
import operator
import os
import unicodedata

import mlflow
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from deepagents import create_deep_agent

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")
EXPERIMENTO = "deep-agent-subagentes"
experimento = mlflow.set_experiment(EXPERIMENTO)
mlflow.langchain.autolog()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

## 2. Ferramentas

- `consultar_vendas`: devolve as vendas de um mês. Os dados são simulados.
- `calcular`: faz contas com números e as quatro operações, usando `ast`. Nunca use `eval` com texto gerado por LLM.

In [2]:
VENDAS_API = {
    "janeiro": {"cafe": 120, "pao": 300, "queijo": 45},
    "fevereiro": {"cafe": 135, "pao": 280, "queijo": 50},
    "marco": {"cafe": 150, "pao": 310, "queijo": 38},
}


@tool
def consultar_vendas(mes: str) -> str:
    """Consulta as vendas de um mês do primeiro trimestre. Retorna a quantidade vendida de cada produto em JSON. Meses válidos: janeiro, fevereiro, março."""
    chave = unicodedata.normalize("NFKD", mes.lower().strip()).encode("ascii", "ignore").decode()
    dados = VENDAS_API.get(chave)
    if dados is None:
        return json.dumps({"mes": mes, "erro": "mês não encontrado"}, ensure_ascii=False)
    return json.dumps({"mes": mes, "vendas": dados}, ensure_ascii=False)


OPERACOES = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}


def _avaliar(no: ast.AST) -> float:
    if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
        return no.value
    if isinstance(no, ast.BinOp) and type(no.op) in OPERACOES:
        return OPERACOES[type(no.op)](_avaliar(no.left), _avaliar(no.right))
    if isinstance(no, ast.UnaryOp) and isinstance(no.op, ast.USub):
        return -_avaliar(no.operand)
    raise ValueError("expressão não permitida")


@tool
def calcular(expressao: str) -> str:
    """Calcula uma expressão aritmética com números, + - * / e parênteses. Exemplo: '(120 + 135) * 2'."""
    return str(round(_avaliar(ast.parse(expressao, mode="eval").body), 2))


FERRAMENTAS = [consultar_vendas, calcular]

print(consultar_vendas.name, "|", calcular.name)

consultar_vendas | calcular


## 3. Definindo os subagentes

Um subagente é um dicionário com `name`, `description`, `system_prompt` e as `tools` que ele pode usar.

- `pesquisador`: busca os dados e faz as contas. Usado no modo isolado.
- `redator`: não tem ferramentas de dados. Lê o resumo compartilhado e escreve a resposta. Usado no modo compartilhado.

A `description` é o que o principal lê para decidir quando delegar.

In [3]:
SUBAGENTE_PESQUISADOR = {
    "name": "pesquisador",
    "description": "Busca vendas do primeiro trimestre e faz as contas. Use para perguntas sobre totais e rankings de vendas.",
    "system_prompt": (
        "Você pesquisa vendas. Consulte cada mês com consultar_vendas, some com calcular "
        "e responda em português, com o total de unidades de cada produto e o mais vendido. "
        "Use somente os dados retornados por consultar_vendas. Não invente preços nem calcule receita. Seja breve."
    ),
    "tools": FERRAMENTAS,
}

SUBAGENTE_REDATOR = {
    "name": "redator",
    "description": "Escreve a resposta final a partir de um resumo de vendas gravado em arquivo. Use quando o resumo já estiver em /compartilhado/vendas.md.",
    "system_prompt": (
        "Você redige respostas. Leia com read_file o arquivo indicado na tarefa e responda "
        "em português, usando somente os números que estão nele. Seja breve."
    ),
}

## 4. Criando os dois agentes

- **Isolado**: o principal só tem o subagente `pesquisador`. O `system_prompt` manda delegar, sem consultar dados diretamente.
- **Compartilhado**: o principal tem as ferramentas de dados, grava o resumo com `write_file` e delega ao `redator`.

In [4]:
PROMPT_ISOLADO = (
    "Você coordena a análise de vendas. Para perguntas sobre vendas, delegue ao subagente pesquisador "
    "com a tarefa completa. Não consulte os dados você mesmo. Responda com o que o subagente devolver."
)

PROMPT_COMPARTILHADO = (
    "Você coordena a análise de vendas. Primeiro, chame write_todos com um plano curto. "
    "Depois, consulte cada mês com consultar_vendas e some com calcular. "
    "Grave um resumo com os totais em /compartilhado/vendas.md usando write_file. "
    "Por fim, delegue ao subagente redator, indicando o caminho do arquivo."
)

agente_isolado = create_deep_agent(
    model=llm,
    middleware=[TodoListMiddleware()],
    subagents=[SUBAGENTE_PESQUISADOR],
    system_prompt=PROMPT_ISOLADO,
)

agente_compartilhado = create_deep_agent(
    model=llm,
    tools=FERRAMENTAS,
    middleware=[TodoListMiddleware()],
    subagents=[SUBAGENTE_REDATOR],
    system_prompt=PROMPT_COMPARTILHADO,
)

## 5. Executando os dois modos

A mesma pergunta vai para os dois agentes. A função `resumo_do_principal` mostra só as chamadas que aparecem no contexto do principal: o que ele fez diretamente e o que delegou. As chamadas feitas dentro de um subagente não aparecem aqui.

In [5]:
TAREFA = "Qual produto mais vendeu no primeiro trimestre? Dê o total de cada produto."


def resposta_final(estado: dict) -> str:
    for mensagem in reversed(estado["messages"]):
        if isinstance(mensagem, AIMessage) and mensagem.content and not mensagem.tool_calls:
            return mensagem.content
    return ""


def chamadas_do_principal(estado: dict) -> list:
    nomes = []
    for mensagem in estado["messages"]:
        if isinstance(mensagem, AIMessage):
            nomes.extend(chamada["name"] for chamada in mensagem.tool_calls)
    return nomes


estado_isolado = agente_isolado.invoke({"messages": [HumanMessage(content=TAREFA)]})
trace_isolado = mlflow.get_last_active_trace_id()

estado_compartilhado = agente_compartilhado.invoke({"messages": [HumanMessage(content=TAREFA)]})
trace_compartilhado = mlflow.get_last_active_trace_id()

display(Markdown(f"### Isolado\n\n{resposta_final(estado_isolado)}"))
display(Markdown(f"### Compartilhado\n\n{resposta_final(estado_compartilhado)}"))

### Isolado

{
  "year": null,
  "qtr": "Q1",
  "by_product": [
    {"product": "cafe", "units": 405},
    {"product": "pao", "units": 890},
    {"product": "queijo", "units": 133}
  ],
  "top_by_units": {"product": "pao", "units": 890},
  "top_by_revenue": null,
  "sources": ["consultar_vendas:janeiro", "consultar_vendas:fevereiro", "consultar_vendas:março"],
  "assumptions": "Usei somente os dados retornados por consultar_vendas para janeiro, fevereiro e março; não há informação de ano nem de preços/valores, portanto não calculei receita nem identifiquei moeda."
}

### Compartilhado

Totais por produto no 1º trimestre:
- cafe: 405
- pao: 890
- queijo: 133

Produto mais vendido: pao — 890 unidades.

Gravei um resumo em /compartilhado/vendas.md e deleguei ao subagente redator para gerar o relatório final. Deseja que eu abra o relatório que o redator escreveu ou envie o texto final aqui?

## 6. Comparando os contextos

A tabela mostra o que apareceu no contexto do principal em cada modo. No modo isolado, o principal não vê as consultas de vendas, porque elas aconteceram dentro do subagente.

In [6]:
for nome, estado in [("Isolado", estado_isolado), ("Compartilhado", estado_compartilhado)]:
    chamadas = chamadas_do_principal(estado)
    print(f"{nome}")
    print(f"  chamadas no principal: {' → '.join(chamadas)}")
    print(f"  consultas de vendas no principal: {chamadas.count('consultar_vendas')}")
    print(f"  delegou a um subagente: {'task' in chamadas}")
    print()

Isolado
  chamadas no principal: task
  consultas de vendas no principal: 0
  delegou a um subagente: True

Compartilhado
  chamadas no principal: write_todos → consultar_vendas → write_todos → consultar_vendas → write_todos → consultar_vendas → write_todos → calcular → calcular → calcular → write_todos → write_file → write_todos → task → write_todos
  consultas de vendas no principal: 3
  delegou a um subagente: True



## 7. Acompanhando o rastreamento no MLflow

Cada execução virou um trace no experimento `deep-agent-subagentes`. Para ver:

1. Abra um terminal na pasta `notebooks/04-DeepAgents` (o caminho do `mlflow.db` é relativo a ela).
2. Rode:

```bash
uv run --project ../.. mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

3. Abra `http://127.0.0.1:5000` no navegador.
4. Em **Experiments**, escolha `deep-agent-subagentes` e abra a aba **Traces**.
5. No trace isolado, a chamada `task` tem spans filhos: as consultas de vendas aparecem dentro do subagente. No trace compartilhado, as consultas aparecem no nível do principal, antes do `task`.
6. Pressione `Ctrl+C` no terminal para parar a interface.

A célula seguinte lista as ferramentas de cada trace pelo notebook, para comparar.

In [7]:
# Usa os ids guardados logo após cada execução, para pegar só os traces desta rodada.
for nome, trace_id in [("Isolado", trace_isolado), ("Compartilhado", trace_compartilhado)]:
    trace = mlflow.get_trace(trace_id)
    ferramentas = [span.name for span in trace.data.spans if span.span_type == "TOOL"]
    print(f"{nome} ({trace_id})")
    print(f"  ferramentas no trace, em ordem: {', '.join(ferramentas)}")
    print()

Isolado (tr-fa64d9208454b19a460fa8ef41e943e1)
  ferramentas no trace, em ordem: task, consultar_vendas, consultar_vendas, consultar_vendas, calcular, calcular, calcular

Compartilhado (tr-d841a2d08f3e401a4e7d3527c9047872)
  ferramentas no trace, em ordem: write_todos, consultar_vendas, write_todos, consultar_vendas, write_todos, consultar_vendas, write_todos, calcular, calcular, calcular, write_todos, write_file, write_todos, task, read_file, write_todos



## Observações

- Contexto isolado protege o principal de detalhes longos, mas o subagente só sabe o que está no texto da tarefa. Se a tarefa for curta demais, o resultado piora.
- Contexto compartilhado, aqui pelo sistema de arquivos, deixa a informação disponível para quem precisar. Em troca, o principal precisa gravar e o subagente precisa saber onde ler.
- A `description` do subagente decide quando o principal delega. Ela deve dizer o que o subagente faz, não como.
- O resultado pode variar entre execuções. Compare as chamadas do principal e a origem dos números, não só o texto final.